# 경기북부 주말축구 게시판 크롤링
- 대상 게시판: https://cafe.daum.net/skfootball/IxVG
- 목표: 5월 게시글 제목 수집
- 방식: `articles.push()` JS 변수 파싱 (로그인 불필요, 1페이지부터 수집)

In [10]:
import re
import json
import time
import requests
from datetime import datetime
import pandas as pd

In [11]:
GRPID = "1O7ju"
FLDID = "IxVG"  # 경기북부 주말축구

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": f"https://cafe.daum.net/skfootball/{FLDID}"
}

BASE_URL = f"https://cafe.daum.net/_c21_/bbs_list?grpid={GRPID}&fldid={FLDID}"

In [12]:
def _parse_pushes(raw):
    """articles.push(...) 블록에서 dataid / title / created 추출"""
    pushes = re.findall(r'articles\.push\(\{[^}]+\}\)', raw)
    result = []
    for item in pushes:
        dm = re.search(r"dataid:\s*'(\d+)'", item)
        tm = re.search(r"title:\s*'([^']*)'" , item)
        cm = re.search(r"created:\s*'([^']+)'", item)
        if dm and tm:
            created_str = cm.group(1) if cm else None
            try:
                created_dt = datetime.strptime(created_str, '%y.%m.%d') if created_str else None
            except ValueError:
                created_dt = None
            result.append({
                "dataid" : dm.group(1),
                "title"  : json.loads(f'"{ tm.group(1)}"'),
                "created": created_dt,
            })
    return result


def _get_page(page_num, prev_page, first_depth, last_depth):
    url = (
        f"https://cafe.daum.net/_c21_/bbs_list?grpid={GRPID}&fldid={FLDID}"
        f"&page={page_num}&prev_page={prev_page}&listnum=20"
        f"&firstbbsdepth={first_depth}&lastbbsdepth={last_depth}"
    )
    raw = requests.get(url, headers=headers).text
    new_first = re.search(r"firstBbsDepth:\s*'([^']+)'", raw)
    new_last  = re.search(r"lastBbsDepth:\s*'([^']+)'",  raw)
    return (
        _parse_pushes(raw),
        page_num,
        new_first.group(1) if new_first else None,
        new_last.group(1)  if new_last  else None,
    )


def collect_posts(max_pages=10, date_from=None, date_to=None):
    """
    게시글 제목 + 작성일 수집 (1페이지부터 시작).

    Parameters
    ----------
    max_pages : 수집할 최대 페이지 수 (페이지당 20개)
    date_from : 시작일 'YYYY-MM-DD' 문자열 (None이면 제한 없음)
    date_to   : 종료일 'YYYY-MM-DD' 문자열 (None이면 제한 없음)

    Returns
    -------
    list of dict  {"dataid", "title", "created"}
    """
    from_dt = datetime.strptime(date_from, '%Y-%m-%d') if date_from else None
    to_dt   = datetime.strptime(date_to,   '%Y-%m-%d') if date_to   else None

    # 1페이지
    raw0 = requests.get(BASE_URL, headers=headers).text
    all_posts = _parse_pushes(raw0)

    m_first = re.search(r"firstBbsDepth:\s*'([^']+)'", raw0)
    m_last  = re.search(r"lastBbsDepth:\s*'([^']+)'",  raw0)

    if not m_first or not m_last:
        print("1페이지 파싱 실패 — raw0 앞부분 확인:")
        print(raw0[:2000])
        return []

    cur_first = m_first.group(1)
    cur_last  = m_last.group(1)
    cur_page  = 1
    print(f"page 1 — {len(all_posts)}개 수집")

    # 2페이지~
    for _ in range(max_pages - 1):
        time.sleep(1)
        posts, cur_page, cur_first, cur_last = _get_page(
            cur_page + 1, cur_page, cur_first, cur_last
        )
        print(f"page {cur_page} — {len(posts)}개 수집")
        all_posts.extend(posts)

        # 날짜 조기 종료: 수집된 게시글이 모두 from_dt 이전이면 중단
        if from_dt and posts:
            latest_in_page = max((p["created"] for p in posts if p["created"]), default=None)
            if latest_in_page and latest_in_page < from_dt:
                print(f"  → 페이지 최신글이 {from_dt.date()} 이전, 수집 중단")
                break

        if not cur_first:
            print("마지막 페이지 도달")
            break

    print(f"\n총 수집: {len(all_posts)}개")

    if from_dt or to_dt:
        all_posts = [
            p for p in all_posts
            if p["created"] is not None
            and (from_dt is None or p["created"] >= from_dt)
            and (to_dt   is None or p["created"] <= to_dt)
        ]
        print(f"날짜 필터 ({date_from} ~ {date_to}): {len(all_posts)}개")

    return all_posts

In [13]:
# 5월 게시글 수집
posts = collect_posts(max_pages=15, date_from="2026-05-01", date_to="2026-05-31")

df = pd.DataFrame(posts)
df["created"] = pd.to_datetime(df["created"])
df = df.sort_values("created", ascending=False).reset_index(drop=True)
df

page 1 — 20개 수집
page 2 — 20개 수집
page 3 — 20개 수집
page 4 — 20개 수집
page 5 — 20개 수집
page 6 — 20개 수집
page 7 — 20개 수집
page 8 — 20개 수집
page 9 — 20개 수집
page 10 — 20개 수집
page 11 — 20개 수집
page 12 — 20개 수집
page 13 — 20개 수집
page 14 — 20개 수집
  → 페이지 최신글이 2026-05-01 이전, 수집 중단

총 수집: 280개
날짜 필터 (2026-05-01 ~ 2026-05-31): 256개


,dataid,title,created
0,58709,6.21 일 오후 18-20시 김포시종합운동장 매칭구합니다~,2026-05-29
1,58700,5월31일 (일) 16시-18시 김포 걸포구장 초청 합니다!!!,2026-05-29
2,58693,5.31일 일요일 12-14시 구리타워 초청합니다!!!!,2026-05-29
3,58694,06. 13(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하),2026-05-29
4,58695,06. 6(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하),2026-05-29
...,...,...,...
251,58407,5월 10일 초청합니다!!!!,2026-05-01
252,58406,5월3일 일요일 20시 농협대학교 매치초청,2026-05-01
253,58405,[무료초청] 5월 2일 토요일 오후 9-11시 2시간 중랑구립운동장 초청합니다.,2026-05-01
254,58404,[▶초청합니다] 2026.05월3일 일요일 20-22시 강남세곡체육공원,2026-05-01


In [14]:
# 5월 게시글 전체 제목 목록
print(f"총 {len(df)}개")
print()
print("--- 5월 게시글 제목 목록 ---")
for _, row in df.iterrows():
    print(f"{row['created'].date()}  {row['title']}")

총 256개

--- 5월 게시글 제목 목록 ---
2026-05-29  6.21 일 오후 18-20시 김포시종합운동장 매칭구합니다~
2026-05-29  5월31일 (일) 16시-18시 김포 걸포구장 초청 합니다!!!
2026-05-29  5.31일 일요일 12-14시 구리타워 초청합니다!!!!
2026-05-29  06. 13(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하)
2026-05-29  06. 6(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하)
2026-05-29  26년 6월 6일(토) 18-20시 의정부직동구장 초청합니다!
2026-05-29  무료초청 5월 31일 일요일 21-23시 구리타워 무료초청합니다
2026-05-29  6월14일 오전8~12시 인덕대학교 초청합니다.
2026-05-29  6월6일 원능수질 9시~12시 초청합니다
2026-05-29  6월달 일요일 일정 21-23시 구리타워 축구장
2026-05-29  5월 31일 (일) 13-15시 수도공고 (강남구) 매치 초청합니다!
2026-05-29  6월 토요일 10:00-12:00 하남보조구장, 감일축구장 오전경기 초청합니다
2026-05-29  내일 토요일 22시 농협대학교 매치초청
2026-05-29  6월달 매주 일요일 다산체육공원축구장 20시~22시 초청합니다
2026-05-29  5.31 일요일 12-14시 구리타워 초청합니다
2026-05-28  5월31일 일요일 21-23시 구리타워 축구장
2026-05-28  6/14 오전8~12시 인덕대학교 초청합니다.
2026-05-28  5월31일 일요일 다산체육공원축구장 13-16시 초청합니다
2026-05-28  5월 31일 (일) 13-15시 수도공고 매치 초청합니다.
2026-05-28  6월 7일 일 초청합니다
2026-05-28  06. 6(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하)
2026-05-28  06. 13(토) / 20 ~ 22 / 

In [18]:
import re

def extract_venue_candidates(title):
    text = title

    # 날짜 제거: 5월16일, 16일 등
    text = re.sub(r'\d+월\s*\d*일?', '', text)
    text = re.sub(r'\d+일', '', text)

    # 시간 제거: 09시~12시, 오전9시, 오후 등
    text = re.sub(r'\d+시\s*[~\-]\s*\d+시', '', text)
    text = re.sub(r'오전|오후', '', text)
    text = re.sub(r'\d+시', '', text)

    # 요일 제거
    text = re.sub(r'월요일|화요일|수요일|목요일|금요일|토요일|일요일', '', text)

    # 구장 번호 제거: '1구장', '2구장' → 숫자만 떼고 앞 장소명 유지
    text = re.sub(r'\s*\d+구장', '', text)

    # 흔한 비장소 단어 제거 (긴 것 먼저)
    stopwords = [
        '무료초청', '매치초청', '초청경기', '초청매치', '초청',
        '신청합니다', '신청바랍니다', '신청',
        '경기모집', '경기신청', '경기',
        '주말리그', '주말', '리그',
        '구합니다', '구인', '모집합니다', '모집',
        '합니다', '입니다', '안내', '공지', '친선', '매칭',
        '상대팀', '상대', '팀', '쿼터', '이상', '무관',
    ]
    for w in stopwords:
        text = text.replace(w, '')

    # 단독 날짜/기간 단어 제거 (다른 단어에 포함된 건 유지: 일산, 월곡 등)
    text = re.sub(r'\b[일월달]\b', '', text)

    # '하', '하하', '하하하' 단독 단어만 제거
    text = re.sub(r'\b하{1,3}\b', '', text)

    # 특수문자 제거 (공백으로 치환)
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)

    # 숫자 제거
    text = re.sub(r'\d+', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text if text else None

df["venue_candidate"] = df["title"].apply(extract_venue_candidates)
df[["title", "venue_candidate"]].head(20)

,title,venue_candidate
0,6.21 일 오후 18-20시 김포시종합운동장 매칭구합니다~,김포시종합운동장
1,5월31일 (일) 16시-18시 김포 걸포구장 초청 합니다!!!,김포 걸포구장
2,5.31일 일요일 12-14시 구리타워 초청합니다!!!!,구리타워
3,06. 13(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하),토 의정부 직동구장 매치
4,06. 6(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하),토 의정부 직동구장 매치
5,26년 6월 6일(토) 18-20시 의정부직동구장 초청합니다!,년 토 의정부직동구장
6,무료초청 5월 31일 일요일 21-23시 구리타워 무료초청합니다,구리타워
7,6월14일 오전8~12시 인덕대학교 초청합니다.,인덕대학교
8,6월6일 원능수질 9시~12시 초청합니다,원능수질
9,6월달 일요일 일정 21-23시 구리타워 축구장,일정 구리타워 축구장


In [21]:
df[["title", "venue_candidate"]].tail(20)

,title,venue_candidate
236,5/10 (일) 06~08 은평구립축구장 초청합니다,은평구립축구장
237,5월달 일요일 다산체육공원축구장 13-16시 초청합니다,다산체육공원축구장
238,파주시 5/9 토 7-10시 교하체육공원 1구장,파주시 토 교하체육공원
239,5월 10일(일) 20시~22시 남양주시 체육문화센터 B구장 11VS11 초청합니다...,남양주시 체육문화센터 B구장 VS
240,5월 10일(일) 18-20 성내유수지 초청합니다.,성내유수지
241,5월 10일 일요일 18:00-20:00 하남보조구장 초청합니다!!,하남보조구장
242,5월달 일요일 다산체육공원축구장 13-16시 초청합니다,다산체육공원축구장
243,5월 10일 일요일 18:00-20:00 하남보조구장 초청합니다!!,하남보조구장
244,5월3일 일요일 농협대학교 20시~22시 필드 키퍼 용병 팀원 모집,농협대학교 필드 키퍼 용병 원
245,5월9.23일 초청합니다.,NaN


In [23]:
# 장소명 → 시 수준 매핑 (키워드 포함 여부로 판단)
CITY_KEYWORDS = {
    "고양시": [
        "교하", "일산", "탄현", "능곡", "화정", "행신", "풍산", "장항",
        "대화", "백석", "마두", "주엽", "정발산", "원흥", "삼송", "지축",
        "향동", "고양", "덕양", "강매", "현천", "토당", "화전",
    ],
    "파주시": [
        "운정", "교하", "야당", "금촌", "문산", "파주", "적성", "탄현",
        "원능", "원릉", "조리", "월롱", "법원", "광탄", "장단",
    ],
    "의정부시": [
        "직동", "의정부", "호원", "장암", "신곡", "가능", "녹양", "민락",
        "고산", "용현", "흥선", "금오", "낙양",
    ],
    "양주시": [
        "양주", "덕계", "덕정", "회천", "옥정", "고읍", "남면", "은현",
    ],
    "포천시": [
        "포천", "소흘", "군내", "가산", "신북",
    ],
    "동두천시": [
        "동두천", "지행", "소요", "생연", "보산",
    ],
    "남양주시": [
        "남양주", "다산", "별내", "진접", "오남", "화도", "퇴계원", "와부",
    ],
    "구리시": [
        "구리", "인창", "갈매", "교문", "수택",
    ],
    "가평군": [
        "가평", "청평", "상면", "조종",
    ],
    "연천군": [
        "연천", "전곡", "청산",
    ],
}

def guess_city(venue):
    if not venue:
        return "미확인"
    for city, keywords in CITY_KEYWORDS.items():
        for kw in keywords:
            if kw in venue:
                return city
    return "미확인"

# unique 장소 후보 추출
unique_venues = (
    df["venue_candidate"]
    .dropna()
    .str.strip()
    .replace("", None)
    .dropna()
    .unique()
)

df_venues = pd.DataFrame({"venue_candidate": sorted(unique_venues)})
df_venues["city"] = df_venues["venue_candidate"].apply(guess_city)
df_venues.head(10)

,venue_candidate,city
0,am 항공대학교,미확인
1,간 중랑구립운동장,미확인
2,감일공원운동장,미확인
3,감일동 축구장,미확인
4,감일동축구장,미확인
5,감일축구장,미확인
6,감일축구장 마감,미확인
7,강남세곡체육공원,미확인
8,고양고등학교,고양시
9,고양시 매치,고양시


In [24]:
output_path = r"e:\git-younghyun\analysis_repo\app_creator\venues_manual_review.xlsx"

df_venues.to_excel(output_path, index=False)
print(f"저장 완료: {output_path}")
print(f"총 {len(df_venues)}개 장소")

저장 완료: e:\git-younghyun\analysis_repo\app_creator\venues_manual_review.xlsx
총 127개 장소


### 수작업 진행함 - 맵핑

In [ ]:
# 수동 검토된 엑셀 불러오기
df_manual = pd.read_excel(r"e:\git-younghyun\analysis_repo\app_creator\venues_manual_review.xlsx")

# venue_confirm이 없거나 '-'인 행 제거
df_manual = df_manual[
    df_manual["venue_confirm"].notna() &
    (df_manual["venue_confirm"].astype(str).str.strip() != "-")
]

# venue_confirm 기준 unique 추출
unique_confirmed = df_manual["venue_confirm"].astype(str).str.strip().unique()

# city 매핑 (남양주 > 양주 순서로 구체적인 것 먼저 체크)
CITY_KEYWORDS_FIXED = {
    "남양주시": ["남양주", "다산", "별내", "진접", "오남", "화도", "퇴계원", "와부", "진건"],
    "고양시":   ["일산", "탄현", "능곡", "화정", "행신", "풍산", "장항", "대화", "백석",
                 "마두", "주엽", "정발산", "원흥", "삼송", "지축", "향동", "고양", "덕양",
                 "강매", "현천", "토당", "화전", "충장"],
    "파주시":   ["운정", "교하", "야당", "금촌", "문산", "파주", "원능", "원릉",
                 "조리", "월롱", "법원", "광탄"],
    "의정부시": ["직동", "의정부", "호원", "장암", "신곡", "가능", "녹양", "민락",
                 "고산", "용현", "흥선", "금오", "낙양", "경민대", "활기"],
    "양주시":   ["양주", "덕계", "덕정", "회천", "옥정", "고읍", "고덕"],
    "포천시":   ["포천", "소흘", "군내", "가산", "신북"],
    "동두천시": ["동두천", "지행", "소요", "생연", "보산"],
    "구리시":   ["구리", "인창", "갈매", "교문", "수택", "왕숙"],
    "가평군":   ["가평", "청평"],
    "연천군":   ["연천", "전곡"],
}

def guess_city_fixed(venue):
    if not venue:
        return "미확인"
    for city, keywords in CITY_KEYWORDS_FIXED.items():
        for kw in keywords:
            if kw in venue:
                return city
    return "미확인"

df_result = (
    pd.DataFrame({"venue_confirm": sorted(unique_confirmed)})
    .assign(city=lambda d: d["venue_confirm"].apply(guess_city_fixed))
)

print(f"총 unique 장소: {len(df_result)}개")
df_result.head(10)

총 unique 장소: 78개


,venue_confirm,city
0,YMCA 축구장,미확인
1,감일공원운동장,미확인
2,감일동 축구장,미확인
3,감일동축구장,미확인
4,감일축구장,미확인
5,강남세곡체육공원,미확인
6,걸포구장,미확인
7,경민대학교 인조잔디구장,의정부시
8,경복대학교 운동장,미확인
9,고덕구장,양주시


In [26]:
df_result.tail(10)

,venue_confirm,city
68,하남보조구장,미확인
69,하남보조구장 감일축구장,미확인
70,하남종합운동장,미확인
71,한강둔치축구장,미확인
72,항공대,미확인
73,항공대학교,미확인
74,항공대학교 운동장,미확인
75,활기구장,의정부시
76,활기체육공원,의정부시
77,활기체육공원 축구장,의정부시


In [27]:
output_path2 = r"e:\git-younghyun\analysis_repo\app_creator\venues_final.xlsx"

df_result.to_excel(output_path2, index=False)
print(f"저장 완료: {output_path2}")
print(f"총 {len(df_result)}개 장소")

저장 완료: e:\git-younghyun\analysis_repo\app_creator\venues_final.xlsx
총 78개 장소


In [31]:
# 수정된 엑셀 불러오기
df_final = pd.read_excel(r"e:\git-younghyun\analysis_repo\app_creator\venues_final.xlsx")

# conflict_name이 있는 행만 사용 (NaN인 행 제외)
df_final = df_final[df_final["conflict_name"].notna()].copy()
df_final["std_name"] = df_final["conflict_name"].astype(str).str.strip()

# unique 통일 구장명
unique_std = sorted(df_final["std_name"].unique())

# 구장명별 city 직접 매핑
VENUE_CITY = {
    "YMCA 축구장":                               "고양시",
    "감일축구장":                                 "하남시",
    "강남세곡체육공원축구장":                      "서울특별시",
    "걸포중앙공원축구장":                          "김포시",
    "경민대학교대운동장":                          "의정부시",
    "경복대학교 남양주캠퍼스운동장":               "남양주시",
    "고양고등학교운동장":                          "고양시",
    "교하체육공원 인조잔디축구장":                 "파주시",
    "구리시민스포츠센터축구장":                    "구리시",
    "구리왕숙체육공원 축구장":                     "구리시",
    "금남축구장":                                 "남양주시",
    "김포시종합운동장":                            "김포시",
    "남양주체육문화센터(종합운동장)메인스타디움":   "남양주시",
    "남양주체육문화센터(종합운동장)축구장 A구장":   "남양주시",
    "남양주체육문화센터(종합운동장)축구장 B구장":   "남양주시",
    "남양주체육문화센터(종합운동장)축구장 C구장":   "남양주시",
    "농협대학교운동장":                            "고양시",
    "다락원체육공원축구장":                        "의정부시",
    "다산체육공원 축구장":                         "남양주시",
    "대화동레포츠공원인조잔디축구장":              "고양시",
    "둔치축구":                                    "고양시",
    "먹골구장":                                   "남양주시",
    "문원체육공원운동장":                          "과천시",
    "불암산스포츠타운축구장":                      "서울특별시",
    "서울어린이대공원잔디축구장":                  "서울특별시",
    "성내유수지축구장":                            "서울특별시",
    "수도전기공업고등학교축구장":                  "서울특별시",
    "신내차량기지축구장":                          "서울특별시",
    "아차산배수지체육공원축구장":                  "서울특별시",
    "연세대학교 신촌캠퍼스대운동장":               "서울특별시",
    "와부공설운동장":                              "남양주시",
    "용마폭포공원축구장":                          "서울특별시",
    "운정건강공원(가온) 인조잔디축구장":            "파주시",
    "원능수질복원센터":                            "파주시",
    "은평구립축구장":                              "서울특별시",
    "인덕대학교운동장":                            "서울특별시",
    "진건푸른물센터인조잔디구장":                  "남양주시",
    "중랑구립잔디운동장":                          "서울특별시",
    "지금푸른물센터축구장":                        "구리시",
    "지영체육공원축구장":                          "고양시",
    "직동근린공원축구장":                          "의정부시",
    "초안산스포츠타운축구장":                      "서울특별시",
    "충장근린체육공원축구장":                      "고양시",
    "파주스타디움보조경기장":                      "파주시",
    "하남종합운동장 주경기장":                     "하남시",
    "하남종합운동장보조경기장":                    "하남시",
    "한국항공대학교운동장":                        "고양시",
    "활기체육공원축구장":                          "의정부시",
}

df_std = pd.DataFrame({
    "std_name": unique_std,
    "city": [VENUE_CITY.get(v, "미확인") for v in unique_std],
})

# 미확인 잔여 확인
missing = df_std[df_std["city"] == "미확인"]
if len(missing):
    print(f"⚠ 미확인 {len(missing)}개: {missing['std_name'].tolist()}")
else:
    print("✓ 미확인 없음")

print(f"\n총 unique 통일 구장명: {len(df_std)}개")
print("\n--- head(10) ---")
display(df_std.head(10))
print("--- tail(10) ---")
display(df_std.tail(10))

✓ 미확인 없음

총 unique 통일 구장명: 48개

--- head(10) ---


,std_name,city
0,YMCA 축구장,고양시
1,감일축구장,하남시
2,강남세곡체육공원축구장,서울특별시
3,걸포중앙공원축구장,김포시
4,경민대학교대운동장,의정부시
5,경복대학교 남양주캠퍼스운동장,남양주시
6,고양고등학교운동장,고양시
7,교하체육공원 인조잔디축구장,파주시
8,구리시민스포츠센터축구장,구리시
9,구리왕숙체육공원 축구장,구리시


--- tail(10) ---


,std_name,city
38,지영체육공원축구장,고양시
39,직동근린공원축구장,의정부시
40,진건푸른물센터인조잔디구장,남양주시
41,초안산스포츠타운축구장,서울특별시
42,충장근린체육공원축구장,고양시
43,파주스타디움보조경기장,파주시
44,하남종합운동장 주경기장,하남시
45,하남종합운동장보조경기장,하남시
46,한국항공대학교운동장,고양시
47,활기체육공원축구장,의정부시


In [32]:
df_std

,std_name,city
0,YMCA 축구장,고양시
1,감일축구장,하남시
2,강남세곡체육공원축구장,서울특별시
3,걸포중앙공원축구장,김포시
4,경민대학교대운동장,의정부시
5,경복대학교 남양주캠퍼스운동장,남양주시
6,고양고등학교운동장,고양시
7,교하체육공원 인조잔디축구장,파주시
8,구리시민스포츠센터축구장,구리시
9,구리왕숙체육공원 축구장,구리시


In [34]:
# venue_confirm → (std_name, city) 룩업 생성
confirm_to_dim = {
    row["venue_confirm"]: (row["std_name"], VENUE_CITY.get(row["std_name"], None))
    for _, row in df_final.iterrows()
}

# 긴 것부터 매칭 (짧은 키워드 오탐 방지)
sorted_confirms = sorted(confirm_to_dim.keys(), key=len, reverse=True)

def match_dim(candidate):
    if not candidate or pd.isna(candidate):
        return None, None
    for confirm in sorted_confirms:
        if confirm in candidate or candidate in confirm:
            return confirm_to_dim[confirm]
    return None, None

df[["std_name", "city"]] = df["venue_candidate"].apply(
    lambda c: pd.Series(match_dim(c))
)

print(f"매핑 성공: {df['city'].notna().sum()}건 / 전체 {len(df)}건")
df[["title", "venue_candidate", "std_name", "city"]].head(20)

매핑 성공: 215건 / 전체 256건


,title,venue_candidate,std_name,city
0,6.21 일 오후 18-20시 김포시종합운동장 매칭구합니다~,김포시종합운동장,김포시종합운동장,김포시
1,5월31일 (일) 16시-18시 김포 걸포구장 초청 합니다!!!,김포 걸포구장,걸포중앙공원축구장,김포시
2,5.31일 일요일 12-14시 구리타워 초청합니다!!!!,구리타워,구리시민스포츠센터축구장,구리시
3,06. 13(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하),토 의정부 직동구장 매치,직동근린공원축구장,의정부시
4,06. 6(토) / 20 ~ 22 / 의정부 직동구장 매치 초청합니다.(하하하),토 의정부 직동구장 매치,직동근린공원축구장,의정부시
5,26년 6월 6일(토) 18-20시 의정부직동구장 초청합니다!,년 토 의정부직동구장,직동근린공원축구장,의정부시
6,무료초청 5월 31일 일요일 21-23시 구리타워 무료초청합니다,구리타워,구리시민스포츠센터축구장,구리시
7,6월14일 오전8~12시 인덕대학교 초청합니다.,인덕대학교,인덕대학교운동장,서울특별시
8,6월6일 원능수질 9시~12시 초청합니다,원능수질,원능수질복원센터,파주시
9,6월달 일요일 일정 21-23시 구리타워 축구장,일정 구리타워 축구장,구리시민스포츠센터축구장,구리시
